# **Forelesning 4 - Språkmodellen Whisper**
I denne forelesningen skal vi se nærmere på bruk av Whisper modellen for bruk til tale-gjenkjenning, audio-til-tekst og noen kjekke anvendelser.

Vi kommer til å bruke en løsning av Whisper modellen, som krever lite kode. Da modellene er *pre-trained* og kan leses mer om på [Hugging face](https://huggingface.co/collections/openai/whisper-release-6501bba2cf999715fd953013).

## **Om Whisper-modellen**
Whisper er en banebrytende språkmodell utviklet av OpenAI, spesielt designet for å transkribere og oversette taledata til tekst. Modellen ble utgitt i september 2022 og er kjent for sine evner innen *automatisk talegjenkjenning* (ASR). Den har en åpen kildekode og støtter en rekke språk og dialekter, noe som gjør den svært allsidig for ulike bruksområder.

### **Nøkkelfunksjoner ved Whisper-modellen**
* *Flerspråklig*: Whisper er trent på et stort datasett som inneholder lyd fra flere språk, og kan derfor håndtere transkripsjon og oversettelse på tvers av mange ulike språk. F.eks. så har vi muligheten til å tolke norsk.

* *Fleksibilitet*: Modellen kan transkribere både taletil-tekst og oversette mellom språk samtidig. Dette gjør den nyttig for applikasjoner som tolkning, teksting, eller språklæring.

* *Robusthet*: Modellen er trent på store mengder mangfoldig data, inkludert bakgrunnsstøy, aksenter, og ustrukturert tale, noe som gir den evnen til å håndtere komplekse og uforutsigbare situasjoner.

* *Flere versjoner*: Whisper kommer i flere størrelser og kompleksitetsnivåer, fra små og raske modeller (som "tiny" og "base") til store og mer nøyaktige modeller (som "large"). Valget av modell avhenger av behovene dine, enten det handler om hastighet eller presisjon.

* *Bruksområder*: Whisper brukes i mange kontekster, som automatiserte transkripsjonssystemer, taleassistanter, og språklige grensesnitt. Den er også populær blant utviklere for å bygge applikasjoner som trenger pålitelig ASR-teknologi.

![Beskrivelse av Whisper](https://raw.githubusercontent.com/openai/whisper/main/approach.png)

# **Hva er tokens?**

* Èn **token** er et ord, eller en del av et ord, som brukes for å *bryte ned* naturlig språk til håndterbare *chunks*.
* Et eksempel kan være setningen "*Jeg liker å spise epler*", og den kan dele i flere tokens som f.eks. slik:
  * "Jeg"
  * "liker"
  * "å"
  * "spise"
  * "epler"
  * "."

Videre kan ord som "liker" bli delt opp igjen til ulike tokener som "lik" og "er". Legg merke til at at det kan være ord, deler av ord eller til og med symbol (som komma, punktum, spørsmåltegn... ).

Dette benytter Whisper modellen, når den skal prosessere audio-data (GPT bruker også denne metodikken). Dette øker effektiviteten, fordi modellen trenger ikke å lagre alle mulige ord i sitt *vokabular*, men kan *bygge sitt vokabular* på mindre enheter, altså disse tokenene.

### Chat GPT
Språkmodeller, ofte kalt LLM (Large Language Models) bruker tokens som grunnleggende enheter for å forstå og generere ny tekst. Når du skriver et setning til GPT (et såkalt *promt*), brytes dette ned til mindre enheter (kalt *tokens*), som modellen kan forstå.

### **Whisper**
Whsiper-modellen er en automatisk talegjenkjenningsmodell, og tar inn tale/lyd (som f.eks. .wav eller .mp3 fil). Første blir lyden konvertert til et spektrogram, som er en visuell representasjon av lydens frekvenser. Deretter blir denne informasjonen gjort om til tokens, altså som tekst.

Altså, sier du til Whisper-modellen "Jeg elsker cola.*", kan det bli "tokenisert" om til:
* "Jeg"
* "elsker"
* "cola"
* "."


# Vi starter med et enkelt eksempel

In [1]:
# Her kan dere laste ned eksempeldata fra Nasjonalbiblioteket blant annet - Kongens tale.
!wget -N https://github.com/NbAiLab/nb-whisper/raw/main/audio/king.mp3
!wget -N https://github.com/NbAiLab/nb-whisper/raw/main/audio/erna.mp3
!wget -N https://github.com/NbAiLab/nb-whisper/raw/main/audio/knuthamsun.mp3

# Installer nødvendig bibliotek.
!pip install transformers #>=4.35.2

--2026-02-01 13:16:33--  https://github.com/NbAiLab/nb-whisper/raw/main/audio/king.mp3
Resolving github.com (github.com)... 140.82.114.3
Connecting to github.com (github.com)|140.82.114.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/NbAiLab/nb-whisper/main/audio/king.mp3 [following]
--2026-02-01 13:16:34--  https://raw.githubusercontent.com/NbAiLab/nb-whisper/main/audio/king.mp3
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1368336 (1.3M) [audio/mpeg]
Saving to: ‘king.mp3’

king.mp3            100%[===================>]   1.30M  --.-KB/s    in 0.03s   

Last-modified header missing -- time-stamps turned off.
2026-02-01 13:16:34 (37.7 MB/s) - ‘king.mp3’ saved [1368336/1368336]

--2026-02-01

In [2]:
# Importer Audio biblioteket
from IPython.display import Audio

# Spill av MP3 filen for kongens tale
Audio("/content/king.mp3")

In [ ]:
Audio("/content/erna.mp3")

In [3]:
# Nødvendige pakker som må installeres for å kjøre Whisper
!pip install transformers torch

# 1. Nasjonalbiblioteket's versjon av Whisper på norsk

In [3]:
import torch
from transformers import pipeline

# Sjekk om vi har GPU tilgjengelig (mye raskere), ellers bruk CPU
device = "cuda:0" if torch.cuda.is_available() else "cpu"
print(f"Kjører på: {device}")

# Last modellen for å kunne gjøre automatisk tale-gjenkjenning (speech recognition) ved å bruke de ulike nb-whisper-models:

### Mulige modeller - a ulike str --> runtime, computational power etc. ###
# NbAiLab/nb-whisper-tiny
# NbAiLab/nb-whisper-base
# NbAiLab/nb-whisper-small
# NbAiLab/nb-whisper-medium
# NbAiLab/nb-whisper-large

# chunk_length_s=30 er kritisk for Whisper for å håndtere lange filer
asr = pipeline(
    "automatic-speech-recognition",
    model="NbAiLab/nb-whisper-small",
    device=device,
    chunk_length_s=30
)

# Transkriber
# Merk: Vi trenger ofte ikke spesifisere 'language': 'nb' for NbAiLab sine modeller,
# da de allerede er tunet for norsk, men det skader ikke.
transkript = asr(
    "king.mp3",
    generate_kwargs={'task': 'transcribe', 'language': 'no'},
    return_timestamps=True
)

# Print bare teksten
print(transkript["text"])

Kjører på: cuda:0


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/967M [00:00<?, ?B/s]

generation_config.json: 0.00B [00:00, ?B/s]

...

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

preprocessor_config.json:   0%|          | 0.00/339 [00:00<?, ?B/s]

Device set to use cuda:0
Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on it's own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).


 Nordmenn er nordlendinger, trøndere, sørlendinger og folk fra alle andre regioner. Nordmenn er også innvandret fra Afghanistan, Pakistan og Polen, Sverige, Somalia og Syria. Det er ikke alltid så lett å si hvor vi er fra, hvilken nasjonalitet vi tilhører. Det vi kaller hjem, er der hjertet vårt er. Og det kan ikke alltid plasseres innenfor landegrenser. Nordmenn er jenter som er glad i jenter, gutter som er glad i gutter, og jenter og gutter som er glad i hverandre. Nordmenn trommer på Gud, Allah, altet og ingenting. Nordmenn liker Grieg, Kygo, Hellbillies og Kari Bremnes. Med andre ord, Norge er dere, Norge er oss. Mitt største håp for Norge er at vi skal klare å ta vare på hverandre. At vi skal bygge dette landet videre på tillit, fellesskap og raushet.


In [4]:
transkript

{'text': ' Nordmenn er nordlendinger, trøndere, sørlendinger og folk fra alle andre regioner. Nordmenn er også innvandret fra Afghanistan, Pakistan og Polen, Sverige, Somalia og Syria. Det er ikke alltid så lett å si hvor vi er fra, hvilken nasjonalitet vi tilhører. Det vi kaller hjem, er der hjertet vårt er. Og det kan ikke alltid plasseres innenfor landegrenser. Nordmenn er jenter som er glad i jenter, gutter som er glad i gutter, og jenter og gutter som er glad i hverandre. Nordmenn trommer på Gud, Allah, altet og ingenting. Nordmenn liker Grieg, Kygo, Hellbillies og Kari Bremnes. Med andre ord, Norge er dere, Norge er oss. Mitt største håp for Norge er at vi skal klare å ta vare på hverandre. At vi skal bygge dette landet videre på tillit, fellesskap og raushet.',
 'chunks': [{'timestamp': (0.02, 4.92),
   'text': ' Nordmenn er nordlendinger, trøndere, sørlendinger'},
  {'timestamp': (5.54, 8.16), 'text': ' og folk fra alle andre regioner.'},
  {'timestamp': (8.66, 25.64),
   'text

Her viser vi bruk av Whisper-modellen i sin enkleste form, og her tar vi i bruk *whisper-small*. (Vi kunne brukt en av de større, men med Colab og forelesningens formål - holder vi det til *small*-versjonen).

## Vi kan også bruke OpenAI sin modell - som ikke er spesialisert for norsk

In [11]:
# Vi kan også kjøre dette - men her er det ikke spesifisert at vi bruker Nasjonalbibliotekets versjon som
# er spesialisert for norsk.

asr_openai = pipeline("automatic-speech-recognition", model="openai/whisper-small",
                      device=0)  # Ensures it runs on GPU if available

transcript_openai = asr_openai(
    "king.mp3",
    generate_kwargs={"language": "no"},
    return_timestamps = True
)
transcript_openai

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/967M [00:00<?, ?B/s]

generation_config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

preprocessor_config.json: 0.00B [00:00, ?B/s]

Device set to use cuda:0


{'text': ' Nordmenn er nordlænninger, trøndere, sølænninger og folk fra alle andre regioner. Nordmenn er også innvandret fra Afghanistan, Pakistan og Polen, Sverige, Somalia og Syria. Det er ikke alltid så lett å si hvor vi er fra, hvilken nasjonalitet vi tilhører. Det vi kaller hjem er der hjertebord er, og det kan ikke alltid plasseres innfor landegrenser. Nordmenn er jenter som er glad i jenter, gutter som er glad i gutter, og jenter og gutter som er glad i hverandre. Nordmenn tommer på Gud, Allah, Altet og ingenting. Nordmenn liker Grig, Kygo, Helbylis og Cardi Bremnes. Norge er dere. Norge er oss. Mitt største håp for Norge er at vi skal klare å ta vare på hverandre, at vi skal bygge dette landet videre på tillid, fjellesskap og reuset.',
 'chunks': [{'timestamp': (0.0, 8.0),
   'text': ' Nordmenn er nordlænninger, trøndere, sølænninger og folk fra alle andre regioner.'},
  {'timestamp': (8.0, 17.0),
   'text': ' Nordmenn er også innvandret fra Afghanistan, Pakistan og Polen, Sver

### Sjekk om du har GPU aktivert
* Se oppe til høyre, og finn `Endre kjøringstype` og velg **GPU**.

In [ ]:
import torch

print("Torch using GPU:", torch.cuda.is_available())
print("Current device:", torch.cuda.current_device())
print("Device name:", torch.cuda.get_device_name(torch.cuda.current_device()))

# Arbeidsanbefaling
Hvis dere synes at denne modellen er interessant, så er det en veldig interessant artikkel dere bør lese fra OpenAI:


> Radford, A., Kim, J. W., Xu, T., Brockman, G., McLeavey, C., & Sutskever, I. (2022). Robust speech recognition via large-scale weak supervision. arXiv. https://doi.org/10.48550/arXiv.2212.04356

[Link til pdf](https://arxiv.org/pdf/2212.04356)

## Mer lesestoff
Ønsker dere å se på de norske modellene vi har brukt over, bør dere sjekke ut HuggingFace sidene til Nasjonalbiblioteket.

[Link til nettside](https://huggingface.co/NbAiLabBeta/nb-whisper-small)



# Sentimentanalyse


Denne koden utfører sentimentanalyse på tekst for å beregne **polaritet**, altså hvor positiv eller negativ teksten er. En flerspråklig, forhåndstrent transformer-modell brukes til å klassifisere teksten som positiv, nøytral eller negativ. Modellen returnerer sannsynligheter for hver klasse, og polariteten beregnes ved å trekke sannsynligheten for negativt sentiment fra sannsynligheten for positivt sentiment. Resultatet er en numerisk verdi mellom −1 og +1, der negative verdier indikerer negativt språk, positive verdier indikerer positivt språk, og verdier nær 0 indikerer nøytral tekst.


In [15]:
from transformers import pipeline

# Sentimentmodell (multilingual, fungerer bra for norsk)
sentiment_pipe = pipeline(
    "sentiment-analysis",
    model="cardiffnlp/twitter-xlm-roberta-base-sentiment",
    return_all_scores=True
)

def polarity_score(text: str) -> float:
    scores = sentiment_pipe(text)[0]

    score_map = {s["label"]: s["score"] for s in scores}

    polarity = (
        score_map.get("positive", 0)
        - score_map.get("negative", 0)
    )

    return round(polarity, 3)


config.json:   0%|          | 0.00/841 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

Device set to use cuda:0
/usr/local/lib/python3.12/dist-packages/transformers/pipelines/text_classification.py:111: UserWarning: `return_all_scores` is now deprecated,  if want a similar functionality use `top_k=None` instead of `return_all_scores=True` or `top_k=1` instead of `return_all_scores=False`.
  warnings.warn(


In [16]:
tekst = transkript["text"]

polaritet = polarity_score(tekst)

print(f"Polaritet: {polaritet}  (-1 = negativ, +1 = positiv)")

Polaritet: 0.574  (-1 = negativ, +1 = positiv)


# **Tilbake til Kongen's tale**

# 1.1 Det er mulighet for å modifisere bruk av Whisper modellen:

* Vi kan legge til *timestamps*, for *når* ting blir sagt i en mp3 fil.
* Dette gir oss muligheten til å finne ut når i en lang audio-fil sier noe interessant - kan være **veldig** tidsbesparende om du en gang skal transkribere masse data og lurer på når dere snakket om et eller annet tema.

In [17]:
# chunk_length_s = bestemmer hvor lange "chunkene" er i sekunder.
# Whisper tar 28 sekunder, og transkriberer det. Så neste 28 sekunder, og så videre...
asr('king.mp3', return_timestamps=True, chunk_length_s=28)

# Grunnen
# Whisper gir bedre resultat med kortere segment av gangen
# Det unngår også minne-problematikk

Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on it's own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).


{'text': ' Nordmenn er nordlendinger, trøndere, sørlendinger og folk fra alle andre regioner. Nordmenn er også innvandrere fra Afghanistan, Pakistan og Polen, Sverige, Somalia og Syria. Det er ikke alltid så lett å si hvor vi er fra, hvilken nasjonalitet vi tilhører. Det vi kaller hjem, er der hjertet vårt er. Og det kan ikke alltid plasseres innenfor landegrenser. Nordmenn er jenter som er glad i jenter, gutter som er glad i gutter. Og jenter og gutter som er glad i hverandre. Nordmenn tror på Gud, Allah, altet og ingenting. Nordmenn liker Grieg, Hygo,ellbillies og Kari Bremnes. Med andre ord. Norge er dere. Norge er oss. Mitt største håp for Norge er at vi skal klare å ta vare på hverandre, at vi skal bygge dette landet videre på tillit, fellesskap og raushet.',
 'chunks': [{'timestamp': (0.02, 7.68),
   'text': ' Nordmenn er nordlendinger, trøndere, sørlendinger og folk fra alle andre regioner.'},
  {'timestamp': (8.66, 27.69),
   'text': ' Nordmenn er også innvandrere fra Afghanist

Whisper-modellen med `chunk_size_s=28` betyr at Whisper prosesserer data (mp3-filen) med 28-sekunders *chunks* - men gir ut outputtet i mer naturlige tale-segmenter.
Whisper tvinger altså ikke segmentene til å være nøyaktig 28 sekunder på grunn av sin naturlige setningsbaserte segmentering. I outputtet over, kan vi se at det er ganske naturlige setninger i hver `´timestamp´`.

#### **Prøv selv**
Prøv å kjør koden under:

```
transcript = asr("king.mp3", return_timestamps=True, chunk_length_s=30, stride_length_s=5)
```
Hvor:

* chunk_length_s=30: Øker vinduet for prosessering.
* stride_length_s=5: Legger til overlapp mellom *chunks* for å unngå å "miste" ord i kantene av chunks.


# 1.2 Vi kan gi timestamps for hvert enkelt ord
Dette kan være nyttig hvis man leter etter spesifikke ord i et intervju, podkast, film, etc...

In [5]:
# Returnerer timestamps på ord-nivå
transcription = asr("king.mp3", chunk_length_s=28, return_timestamps="word", generate_kwargs={'task': 'transcribe', 'language': 'no'})
transcription

Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on it's own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).


{'text': ' Nordmenn er nordlendinger, trøndere, sørlendinger og folk fra alle andre regioner. Nordmenn er også innvandret fra Afghanistan, Pakistan og Polen, Sverige, Somalia og Syria. Det er ikke alltid så lett å si hvor vi er fra, hvilken nasjonalitet vi tilhører. Det vi kaller hjem, er der hjertet vårt er. Og det kan ikke alltid plasseres innenfor landegrenser. Nordmenn er jenter som er glad i jenter, gutter som er glad i gutter, og jenter og gutter som er glad i hverandre. Nordmenn tror på Gud, Allah, altet og ingenting. liker Grieg, Kygo, Helbillis og Kari Bremnes. Med andre ord, Norge er dere, Norge er oss. Mitt største håp for Norge er at vi skal klare å ta vare på hverandre, at vi skal bygge dette landet videre på tillit, fellesskap og raushet.',
 'chunks': [{'text': ' Nordmenn', 'timestamp': (0.0, 1.42)},
  {'text': ' er', 'timestamp': (1.42, 1.7)},
  {'text': ' nordlendinger,', 'timestamp': (1.7, 3.16)},
  {'text': ' trøndere,', 'timestamp': (3.16, 3.96)},
  {'text': ' sørlen

# Men hva hjelper det her?
Nå kan vi lage en funksjon i Python for å søke etter hvor et spesifikt ord blir brukt.

Potensiell use-case: Du ser gjennom $100$+ timer med opptak for transkripsjon og husker at i et intervju snakket du om noe ekstremt sjeldent og interessant, men kan ikke finne ut hvor det er. I stedet for å høre gjennom $10$+ timer og risikere å gå glipp av det, kan du bruke dette verktøyet. Mens koden kjører, kan du gjøre noe annet! :)

In [6]:
def find_word_timestamps(transcription, word):
    """
    Funksjon som lager timestamps til spesifikke ord i transkripsjonen.

    Parametere:
    - transcription: Dictionary som inneholder transkripsjonsteksten og chuncks med timestamps.
    - word: Ordet man søker etter i transkripsjonen.

    Returns:
    - Liste av timestamps hvor det ordet ('word') befinner seg.
    """
    word = word.lower()  # Konverterer alt til lower-case for å forhindre case-sensitivity.
    timestamps = []

    for chunk in transcription['chunks']:
        # Ekstraher ord og fjern punctuation (punktum, komma etc.)
        chunk_text = chunk['text'].strip().lower().rstrip('.,')
        # Sjekker om chunken inneholder 'word' (ordet vi leter etter)
        if word in chunk_text.split():
            timestamps.append(chunk['timestamp'])

    return timestamps

# Eksempelbruk med asr function
# transcription = asr("path/to/audio/file")
word_to_find = "Bremnes" # Dette skal bare være et ord, ingen komma/kolon/semikolon etc.
timestamps = find_word_timestamps(transcription, word_to_find)

print(f"Timestamps for the word '{word_to_find}':", timestamps)

Timestamps for the word 'Bremnes': [(61.52, 62.38)]


In [7]:
# Test for andre ord
word_to_find = "Nordmenn" # Dette skal bare være et ord, ingen komma/kolon/semikolon etc.
timestamps = find_word_timestamps(transcription, word_to_find)

print(f"Timestamps for the word '{word_to_find}':", timestamps)

Timestamps for the word 'Nordmenn': [(0.0, 1.42), (8.88, 9.34), (41.87, 42.37), (51.13, 51.61)]


# 1.3 Transkribere fra ett språk til et annet språk.
Vi kan også transkribere til nynorsk eller engelsk, som vi viser nedenfor.

In [8]:
# Transcribe to Nynorsk
asr("king.mp3", chunk_length_s=28, generate_kwargs={'task': 'transcribe', 'language': 'nn'})

Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on it's own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).


{'text': ' Nordmenn er nordlendingar, trøndarar, sørlendingar og folk frå alle andre regionar. Nordmenn er også innvandrarar frå Afghanistan, Pakistan og Polen, Sverige, Somalia og Syria. Det er ikkje alltid så lett å seie kvar vi er frå, kva nasjonalitet vi tilhøyrer. Det vi kallar heim, er der hjartet vårt er. Og det kan ikkje alltid plasserast innanfor landegrenser. Nordmenn er jenter som er glad i jenter, gutter som er glad i gutter og jenter og gutter som er glad i kvarandre. Nordmenn trur på Gud, Allah, altet og ingenting. Nordmenn liker Grieg, Kygo, Helbillies og Kari Bremnes. Med andre ord, Norge er dere, Norge er oss. Mitt største håp for Noreg er at vi skal klare å ta vare på kvarandre, at vi skal bygge landet videre på tillit, fellesskap og raushet.'}

In [12]:
# Transcribe to English
asr("king.mp3", chunk_length_s=28, generate_kwargs={'task': 'transcribe', 'language': 'en'})

Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on it's own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).


{'text': ' Norwegians are northerners, trønders, southerners and people from all other regions. Norwegians are also immigrated from Afghanistan, Pakistan and Poland, Sweden, Somalia and Syria. It is not always so easy to say where we are from, what nationality we belong to. What we call home is where our heart is. And it cannot always be placed within national borders. Norwegians are girls who like girls, boys who like boys, and girls and boys who like each other. Norwegians believe in God, Allah, everything and nothing. Norwegians like Grieg, Kygo, Helbillis and Kari Bremnes. In other words, Norway is you, Norway is us. My biggest hope for Norway is that we will be able to take care of each other. That we should build the country further on trust, community and generosity.'}

In [13]:
asr_openai("king.mp3", chunk_length_s=28, generate_kwargs={'task': 'transcribe', 'language': 'de'})

Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on it's own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).


{'text': ' Nurmen ist Nord- und Südländen von allen anderen Regionen. Nurmen ist auch von Afghanistan, Pakistan, Polen, Sverige, Somalia und Syrien eingewandert. Es ist nicht alles so leicht zu sagen, wo wir sind. Welche Nationalität wir hören. Was wir heben, ist das, was unser Herz ist. Und das kann nicht alles in der Landung geplacert werden. Wenn man ein Mädchenacert. Sie sind nicht immer geplacert. Sie sind nicht immer geplacert. Sie sind nicht immer geplacert. Sie sind nicht immer geplacert. Sie sind nicht immer geplacert. Sie sind nicht immer geplacert. Sie sind nicht immer geplacert. Sie sind nicht immer geplacert. Sie sind nicht immer geplacert. Sie sind nicht immer geplacert. Sie sind nicht immer geplacert. Sie sind nicht immer geplacert. Sie sind nicht immer geplacert. Sie sind nicht immer geplacert. Sie sind nicht immer geplacert. Sie sind nicht immer geplacert. Sie sind nicht immer geplacert. Sie sind nicht immer geplacert. Sie sind nicht immer geplacert. Sie sind nicht imm

In [ ]:
asr_openai("ML_Pod_transcript-kopi.wav", chunk_length_s=28, generate_kwargs={'task': 'transcribe', 'language': 'ar'})


/usr/local/lib/python3.11/dist-packages/transformers/models/whisper/generation_whisper.py:512: FutureWarning: The input name `inputs` is deprecated. Please make sure to use `input_features` instead.
  warnings.warn(


{'text': " أنت one of the greatest teachers of machine learning AI ever from CS231N to today. What advice would you give to beginners interested in getting into machine learning? Beginners are often focused on like what to do and I think the focus should be more like how much youصدقائي على أعلى أعلى أعلى 10,000 سنة. أعرف أنه يجب أن تتحرك الأ و ستكتب و ستكتب و ستكتب و ست ويعتقد أنه يتفكير أكثر على أن تستخدمك 10.000 سنة. ويعتقد أنه يتفكير بأن ماذا تستخدم؟ أن تستخدم بأن ماذا تستخدم؟ أن تستخدم بأن ماذا تستخدم؟ أن تستخدم؟ أن تستخدم؟ أن تستخدم؟ أن تستخدم؟ أن تستخدم؟ أن تستخدم؟ أن تستخدم؟ أن تستخدم؟ أن تستخدم؟ أن تستخدم؟ أن تستخدم؟ أن تستخدم؟ أن تستخدم؟ أن تستخدم أساسين من أساسين من أساسين من أساسين من أساسين من أساسين من أساسين من أساسين من أساسين من أساسين من أساسين من أساسين من أساسين من أساسين من أساسين من أساسين من أساسين من أساسين من أساسين من أساسين من أساسين من أساسين من أساسين من أساسين من أساسين من أساسين من أساسين من أساسين من أساسين من أساسين من أك أخبرك أخبرك أخبرك أخبرك أخبرك أخ

## 2. Bruk av OpenAI sin versjon av Whisper direkte
Her bruker vi altså ikke Nasjonalbibliotekets modell, men laster inn whisper for å kjøre det lokalt.

Men, da må man kjøre følgende

```
# Laster ned
!pip install openai-whisper

# Tar i bruk whisper biblioteket
import whisper
```

In [14]:
!pip install openai-whisper

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 15.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for openai-whisper: filename=openai_whisper-20250625-py3-none-any.whl size=803979 sha256=4f0bb448568b07cc4fe34cd61b4d45e62af7a58910678222e606015edb177821
  Stored in directory: /root/.cache/pip/wheels/61/d2/20/09ec9bef734d126cba375b15898010b6cc28578d8afdde5869
Successfully built openai-whisper


In [15]:
import whisper

# Load Whisper model - her uten å spesifisere Nasjonalbiblioteket
model = whisper.load_model("medium")

# Transkriberer audio-filen
result = model.transcribe("/content/ML_Pod_transcript-kopi.wav", language='en') # språk kan være f.eks. 'en', 'no'

# Splitter transkriptet som setninger, for å gjøre det mer leselig.
text = result["text"]

# Definer avslutning av setninger for leseslighet
sentence_endings = '.!?'

# Add line breaks after each sentence-ending punctuation
formatted_text = ""
for char in text:
    formatted_text += char
    if char in sentence_endings:
        formatted_text += '\n'  # Add a newline after each sentence-ending punctuation

# Print ut transkriptet
print(formatted_text)

100%|█████████████████████████████████████| 1.42G/1.42G [00:22<00:00, 66.5MiB/s]


 You're one of the greatest teachers of machine learning, AI, ever.
 From CS231N to today, what advice would you give to beginners interested in getting into machine learning?
 Beginners are often focused on what to do, and I think the focus should be more like how much you do.
 So I am kind of like believer on a high level in this 10,000 hours kind of concept where You just kind of have to just pick the things where you can spend time and you care about and you're interested in.
 You literally have to put in 10,000 hours of work.
 It doesn't even like matter as much like where you put it and you'll iterate and you'll improve and you'll waste some time.
 I don't know if there's a better way.
 You need to put in 10,000 hours.
 But I think it's actually really nice because I feel like there's some sense of determinism about being an expert at a thing if you spend 10,000 hours.
 You can literally pick an arbitrary thing.
 And I think if you spend 10,000 hours of deliberate effort and work

# Oppgaver

## Oppgave 1
Spill inn din egen stemme (bruk pc, mobil eller noe lignende), og last opp i Colab - og prøv å transkriber filen.

## Oppgave 2
List opp noen use-case'r for dette. Kanskje skal du gjøre noe form for kvalitativ studie i masteroppgaven din, og må transkribere masse data?

## Oppgave 3
Lag en funksjon som transformerer en videofil (mp4) til audio-fil (som mp3 eller wav).

Da vil du få bruk for biblioteket `import ffmpeg`. Hva kan en slik funksjon brukes til?

___